# Tutorial 4

To implement value function iteration in the computer, we need a discrete state space and a stationary and "markovian" problem.

In heterogeneous‐agent macro models with incomplete markets, income follows a stochastic process. Until now, we have imposed an ad hoc grid of income states and arbitrary transition probabilities -see Tutorial 2. 

Hence, in this tutorial we will:

1. Introduce discrete Markov Chains

2. Introduce the Tauchen method for discretizing a stochastic income that follows AR(1) process.

### 1. What are Markov chains?

A **Markov chain** is a sequence of random (stochastic) variables

$$
\{s_t\}_{t=0}^{\infty},\qquad s_t\in\mathcal S
$$

with the *finite memory* property

$$
\Pr(s_{t+1}=j\mid s_t=i,\;s_{t-1},\dots,s_0)=\Pr(s_{t+1}=j\mid s_t=i)\equiv P_{ij}.
$$

i.e. the probabilities to move from one state to the other depend only on the last state.

* $\mathcal S$ is the (finite or countable) **state space**.
* $P=(P_{ij})$ is the **transition matrix** between states; each row sums to one.

#### **Irreducibility**

A chain is **irreducible** if every state can be reached from every other state with positive probability in a finite number of steps.
Formally, for all $i,j\in\mathcal S$ there exists $n$ such that $(P^{n})_{ij}>0$.

*Why we care*: irreducibility guarantees that the long‑run distribution does not depend on the initial state—crucial when you want a unique stationary/ergodic distribution to integrate expectations in dynamic programming.

#### **Aperiodicity**

A chain is **aperiodic** if it doesn't cycle in a predictive way.

*Why we care*: aperiodicity, together with irreducibility, implies the chain is **ergodic**—$P^{n}\to\mathbf 1 \pi$ as $n\to\infty$ where $\pi$ is the unique stationary distribution. Value‑function iteration often relies on this convergence to justify replacing expectations with integrals/sums with respect to $\pi$.


**Examples:**

Question: Is the following Markov Chain irreducible and aperiodic?
$$
P = \begin{bmatrix}
0.9   & 0.1   & 0 \\[4pt]
0.1 & 0.8 & 0.1 \\[4pt]
0   & 0.2 & 0.8
\end{bmatrix}
$$

where the first column/row is the state = poor, second column/row is the state = middle class, and the third column/row is the state = rich.

![image](Figures/example_irreducibility.png)

Question: Is the following Markov Chain irreducible and aperiodic?
$$
P = \begin{bmatrix}
0  & 1 & 0 \\[4pt]
0 & 0 & 1 \\[4pt]
1   & 0 & 0
\end{bmatrix}
$$

![image](Figures/example_aperiodicity.png)

### 2. Why Markov structure matters in dynamic programming

In **recursive macroeconomic problems** we want a state vector $s_t$ that contains everything needed to forecast the future. The Bellman equation

$$
V(s)=\max_{a\in\Gamma(s)}\Bigl\{u(s,a)+\beta\,\mathbb E\bigl[V(s')\mid s,a\bigr]\Bigr\}
$$

assumes the conditional distribution $s'\mid s,a$ is well‑defined and time‑invariant. A Markov chain with the properties above delivers exactly that:

* **Finite Memory** → expectations depend only on current state/action.
* **Ergodicity** → guarantees the law of large numbers for simulation and ensures contraction mappings used in value‑function iteration are well behaved.

If the chain were reducible or periodic, you could end up with multiple invariant distributions or cycles, breaking many convergence proofs.


**What is $s$ in our particular case?**

Household's earnings process $y$ (in the lecture also denoted as $e$)

Empirically, individual/household's earnings are often modelled as an **AR(1) with normal shocks**

$$
y_{t+1}=\rho\,y_t+\varepsilon_{t+1},\qquad \varepsilon_{t+1}\sim\mathcal N(0,\sigma^2).
$$

This process is *Markov* but **lives on a continuum**. To see this, consider the definition of the mean of random variable x:

$$ \mathbb{E}(x) = \int_{-\infty}^{\infty} x \phi(x) dx$$

with $\phi(x)$ being the probability density function of x.

In our case, this random variable is evolution of income given current income (depicted by the AR(1) function):

$$ \mathbb{E}(F(y_{t+1})|y_t) = \underbrace{\int_{-\infty}^{\infty} F(y_{t+1}) \phi(y_{t+1}|y_t) dy_{t+1}}_{\text{A CONTINUOUS Function!}}$$

However, computers are discrete machines. Hence, we need to discretize our continouos income process over $N$ grid points $x_i$:

i.e. from continuous into bins:

$$ \Chi_{i=1,...,N}$$

$$ \int_{\Chi_i} F(y_{t+1}) \phi(y_{t+1}|y_t \in \Chi_j) \approx \omega_{ji} F(x_i)$$

where $\omega_{ji} F(x_i)$ comes from using numerical integration (e.g. Gauss-Chebyshev Quadrature)

And here is where **Tauchen (1986) method** comes to rescue.

### 3. The Tauchen discretization algorithm

Tauchen’s method is a way to turn a continuous AR(1) process

$$
y_{t+1} \;=\; \rho\,y_t + \varepsilon_{t+1}, 
\qquad \varepsilon_{t+1}\sim N(0,\sigma_e^2)
$$

into a discrete‐state Markov chain that can be handled by the computer. 

In [ ]:
"""
Tauchen(ρ,N,sigma,μ)

Generate a discrete approximation to an AR(1) process, following Tauchen (1987).

Uses importance sampling: each bin has probability 1/N to realize

# Arguments
- `ρ`: autocorrelation coefficient
- `N`: number of gridpoints
- `sigma`: long-run variance
- `μ`: mean of the AR(1) process

# Returns
- `grid_vec`: state vector grid
- `P`: transition matrix
- `bounds`: bin bounds
"""

using Distributions, LinearAlgebra

function Tauchen(ρ::Float64, N::Int; sigma::Float64 = 1.0, μ::Float64 = 0.0)  
    dis = Normal()
    pr_ij(x, bound1, bound2, ρ, sigma_e) = pdf.(dis, x) .*
         (cdf.(dis, (bound2 - ρ .* x) ./ sigma_e) -
         cdf.(dis, (bound1 - ρ .* x) ./ sigma_e) )
    
        grid_probs = range(0.0, stop = 1.0, length = N+1)   # generate equi-likely bins
    
        bounds = quantile.(dis, grid_probs[1:end])     # corresponding bin bounds
    
        # replace [-]Inf bounds, by finite numbers
        bounds[1] = bounds[2] - 1.0e2
        bounds[end] = bounds[end-1] + 1.0e2
    
        # Calculate grid() - centers
        grid_vec = N * (pdf.(dis, bounds[1:end-1]) - pdf.(dis, bounds[2:end]))
    
        sigma_e = sqrt(1 - ρ^2)# Calculate short run variance
        P = fill(0.0, (N, N)) # Initialize Transition Probability Matrix
    
        for j = 1:N
            p(x) = pr_ij(x,bounds[j], bounds[j+1], ρ, sigma_e)
            for i = 1:floor(Int, (N-1)/2)+1 # Exploit Symmetrie to save running time
                P[i, j] = my_integrate(p, bounds[i], bounds[i+1]) # Evaluate Integral
            end
        end
        # Exploit Symmetrie Part II
        P[floor(Int, (N - 1) / 2) + 2:N, :] = P[(ceil(Int, (N - 1) / 2):-1:1), end:-1:1]
    
    # Make sure P is a Probability Matrix
    P = P ./ sum(P, dims = 2)
    
    grid_vec   = grid_vec .* sigma .+ μ
    lmul!(sigma, bounds)
    # bounds = bounds .* sigma
    
    return grid_vec, P, bounds
    
end

#---------------------------------------------------------
# Integration (Gauss Chebychev)
#---------------------------------------------------------
function my_integrate(f::Function, a::Number, b::Number)
    nodes, weights = my_qnwcheb(500, a, b)
    I = weights'*f.(nodes)
end

function my_qnwcheb(n::Integer, a::Number, b::Number)
    nodes = (b + a) / 2 .- (b - a) / 2 .* cos.(pi / n .* (0.5:(n - 0.5)))
    weights = ((b - a) / n) .* (cos.(pi / n .* ((1:n) .- 0.5) * (2:2:n-1)') *
    (-2.0 ./ ((1:2:n-2) .* (3:2:n))) .+ 1)
    return nodes, weights
end


Here’s the idea of Tauchen's method in four simple steps:

##### Step 1. **Choose the number of “bins” $N$.**
You’ll approximate the continuously valued $y_t$ by one of $N$ grid‐points.

##### Step 2. **Build equally‐probable bins under the stationary distribution.**

* If the AR(1) were run forever, $y_t$ would be $N(\mu,\sigma^2)$ with long‐run variance $\sigma^2$.
* We split the normal distribution into $N$ intervals, each containing probability $1/N$.
* In code:

  ```julia
  grid_probs = range(0.0, stop = 1.0, length = N+1)
  bounds = quantile.(dis, grid_probs)        # bin edges in the standard‐normal world
  bounds[1] = bounds[2] - 1e2                # replace -Inf with a large negative
  bounds[end] = bounds[end-1] + 1e2          # replace +Inf with a large positive
  ```
* After computing everything on the standard normal scale, the function scales both the grid and the bounds by the desired $\sigma$ and shifts by $\mu$.

##### Step 3. **Pick a representative grid value in each bin.**

![image](Figures/representative_element1.png) ![image](Figures/representative_element2.png) ![image](Figures/representative_element3.png)

A simple choice is the midpoint (i.e. middle figure), but one can do better by taking the bin’s expected value under the stationary density.  The code uses the formula

$$
  x_i \;\approx\; N\bigl[\phi(b_i) - \phi(b_{i+1})\bigr]
$$

where $\phi$ is the standard‐normal pdf and $b_i$ are the boundaries.  In Julia:

```julia
grid_vec = N * (pdf.(dis, bounds[1:end-1]) .- pdf.(dis, bounds[2:end]))
```

Finally it rescales and recenters:

```julia
grid_vec .= grid_vec .* sigma .+ μ
```

##### Step 4. **Compute the transition probabilities by integrating the AR(1) kernel over each pair of bins.**

* Given you’re in bin $j$ today (so $y_t$ lies between $b_j$ and $b_{j+1}$), the probability of landing in bin $i$ tomorrow is

  $$
    P_{ij} \;=\;\int_{b_i}^{\,b_{i+1}} \int_{b_j}^{\,b_{j+1}}
      f(y_{t+1}\mid y_t)\,f(y_t)\,dy_t\,dy_{t+1}
    \Big/  \int_{b_j}^{\,b_{j+1}} f(y_t)\,dy_t
  $$

  but by choosing bins of equal probability, the denominator is $1/N$, so we integrate the joint density and then normalize rows at the end.
* The code sets up a helper

  ```julia
  pr_ij(x, bound1, bound2, ρ, σ_e) =
    pdf(dis, x) .* (
      cdf(dis, (bound2 - ρ*x)/σ_e) -
      cdf(dis, (bound1 - ρ*x)/σ_e)
    )
  ```

  which for a “previous” draw $x$ in bin $j$ computes the conditional chance the AR(1) jump lands in $[{\rm bound1},{\rm bound2}]$.
* It then uses Gauss–Chebyshev quadrature (`my_integrate`) to integrate over each bin $i$:

  ```julia
  for j in 1:N
      p(x) = pr_ij(x, bounds[j], bounds[j+1], ρ, sigma_e)
      for i in 1:ceil(Int,(N)/2)  # exploit symmetry
          P[i,j] = my_integrate(p, bounds[i], bounds[i+1])
      end
  end
  # fill in the rest by symmetry, then normalize rows:
  P .= P ./ sum(P, dims=1)'
  ```
* Here `sigma_e = sqrt(1 - ρ^2)` is the AR(1) innovation’s standard deviation on the standard‐normal scale; after all the integrating, the entire matrix is renormalized so each column sums to one (so from each today’s bin you have a bona fide probability distribution over tomorrow’s bins).

Putting it all together, `Tauchen(ρ,N; sigma,μ)` returns:

* **`grid_vec`**: the $N$ representative states (the grid)
* **`P`**: the $N\times N$ transition probability matrix
* **`bounds`**: the edges of each bin

Once you have $(\mathrm{grid\_vec},P)$, you can replace the continuous idisyncratic income shock $y_t$ in your model with this finite‐state Markov chain and solve your model using value‐function iteration, policy iteration, etc.

**Key intuition in simple terms**

* **Grid by equal-probability slices**: cut the long-run distribution into $N$ slices, so each slice occurs with probability $1/N$.
* **Representative points**: choose one “typical” value within each slice.
* **Transition kernel**: compute how likely you are to move from slice $j$ to slice $i$ by integrating the AR(1) normal density over those slices.
* **Result**: a discrete Markov chain that closely mimics your original AR(1).

With this at hand, let as take the two period model framework we explored in Tutorial 2 and use a realistic income process.

In [ ]:
### Income process discretization and Markov Chain Simulation
# For this exercise we use the same two period model framework seen in the previous tutorial.

# Packages
using Roots, Plots
using KernelDensity

## Parameters
m_par = (α = 0.25, β = 0.98, π = 0.5, ξ = 4.0, ρ = 0.98)

## Discretize AR(1) income process using Tauchen.
include("Tauchen.jl")

ny = 7  # Specify the number of income states

grid_y, Π, bounds = Tauchen(m_par.ρ,ny)
grid_y = exp.(grid_y)

# # Alternatively you can write:
# grid_y = exp.(Tauchen(ρ,ny)[1])
# Π = Tauchen(ρ,ny)[2]

## Household
# CRRA utility function
util(c) = (c .^ (1.0 .- m_par.ξ) .- 1.0) ./ (1.0 .- m_par.ξ)
mutil(c) = c .^ (-m_par.ξ)

# Partial equilibrium so prices are given.
w = 1.0
global R = 1.03

Astar = zeros(ny)

for i in eachindex(grid_y)
    Savings(a) = mutil(w * grid_y[i] .- a) .- m_par.β .* R .* (Π[i,:]' * mutil(R.*a .+ w .* grid_y))
    
    # Find the root of the Euler equation --> Policy function for Savings
    Astar[i] = find_zero(Savings, (- (w / R) * grid_y[1], w))
end

# Find consumption in the second period --> Policy function for Consumption
C_2 = R * Astar .+ w .* grid_y

## Plot Policy Functions
figure1 = plot(grid_y,Astar,label=nothing) # Plot policy functions
title!("Policy Function for Savings") # Title of the graph
xlabel!("Income")
ylabel!("Saving")
display(figure1)

figure2 = plot(grid_y,C_2,labels=nothing) # Plot policy functions
title!("Policy Function for Consumption") # Title of the graph
xlabel!("Income")
ylabel!("Consumption")
display(figure2)

## Markov Chain Simulation
using Random

H = 10000 # Number of households to simulate
T = 2     # Model with only two periods

state = rand(1:ny,H,T)           # Initialize vector of states (i.e. establish some initial state s_0 = i)
PP = cumsum(Π,dims = 2)          # Cummulative Transition matrix # cumsum along columns

unif_dist_var = rand(H)

Savings_h = zeros(H)
Consumption_h = zeros(H)

for h = 1:H
    state[h,2] = count(PP[state[h,1],:].< unif_dist_var[h])+1   # Update states
    Savings_h[h] = Astar[state[h,1]]
    Consumption_h[h] = R * Astar[state[h,1]] .+ w .* grid_y[state[h,2]]
end

kd = kde(Savings_h, bandwidth=0.02)
histogram(Savings_h, bins=20, normalize=:pdf, label="Simulated Histogram", title="Distribution of Savings", xlabel="Savings", ylabel="Density")
plot!(kd.x, kd.density, label="Kernel Density", lw=2)